# MAPPO Training

Multi-Agent Proximal Policy Optimization for the  
LLM-Empowered Multi-UAV Active Sensing prototype.

**2 UAVs | 4x3 grid | 12 sectors | 30 days per episode**

### Files needed (upload as Kaggle dataset)
```
uav_env.py
networks.py
simulation_log.csv
grid_config.json
```

## 1. Setup

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import deque

# ── Kaggle paths (adjust if running locally) ──────────────────────────────────
# When running on Kaggle, dataset files are at /kaggle/input/<dataset-name>/
# Change these paths to match your Kaggle dataset name

import sys
if os.path.exists('/kaggle/input'):
    BASE = '/kaggle/input/uav-crop-disease'   # <- change to your dataset name
    sys.path.insert(0, BASE)
else:
    BASE = '..'
    sys.path.insert(0, '.')

SIM_LOG_PATH    = os.path.join(BASE, 'simulation', 'simulation_log.csv') if not os.path.exists('/kaggle/input') else os.path.join(BASE, 'simulation_log.csv')
GRID_CFG_PATH   = os.path.join(BASE, 'grid', 'grid_config.json')         if not os.path.exists('/kaggle/input') else os.path.join(BASE, 'grid_config.json')

from uav_env import UAVFieldEnv
from networks import ActorNetwork, CriticNetwork

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Sim log: {SIM_LOG_PATH}')
print(f'Grid   : {GRID_CFG_PATH}')

## 2. Hyperparameters

In [ ]:
# Training
N_EPISODES      = 500
K_EPOCHS        = 10       # PPO update epochs per episode
GAMMA           = 0.99     # discount factor
GAE_LAMBDA      = 0.95     # GAE smoothing
EPS_CLIP        = 0.2      # PPO clip range
ENTROPY_COEFF   = 0.01     # entropy bonus (encourages exploration)
LR_ACTOR        = 3e-4
LR_CRITIC       = 3e-4

N_UAVS          = 2
OBS_SIZE        = 17
JOINT_SIZE      = 34       # OBS_SIZE * N_UAVS

SAVE_INTERVAL   = 100      # save model weights every N episodes
LOG_INTERVAL    = 20       # print progress every N episodes

## 3. Environment & Networks

In [ ]:
env = UAVFieldEnv(SIM_LOG_PATH, GRID_CFG_PATH)

# Two local actors (one per UAV) + one centralized critic
actors  = [ActorNetwork().to(DEVICE) for _ in range(N_UAVS)]
critic  = CriticNetwork().to(DEVICE)

actor_optimizers = [torch.optim.Adam(a.parameters(), lr=LR_ACTOR)
                    for a in actors]
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=LR_CRITIC)

total_params = sum(sum(p.numel() for p in a.parameters()) for a in actors) + \
               sum(p.numel() for p in critic.parameters())
print(f'Total trainable parameters: {total_params:,}')
print(f'Environment T={env.T} steps per episode')

## 4. Rollout Buffer

In [ ]:
class RolloutBuffer:
    """
    Stores one full episode of experience for all UAVs.
    """
    def __init__(self):
        self.clear()

    def clear(self):
        self.obs       = [[] for _ in range(N_UAVS)]   # obs per UAV
        self.actions   = [[] for _ in range(N_UAVS)]
        self.log_probs = [[] for _ in range(N_UAVS)]
        self.rewards   = [[] for _ in range(N_UAVS)]
        self.values    = []                             # critic values (joint)
        self.dones     = []

    def store(self, obs_list, actions_list, log_probs_list,
              rewards_list, value, done):
        for u in range(N_UAVS):
            self.obs[u].append(obs_list[u])
            self.actions[u].append(actions_list[u])
            self.log_probs[u].append(log_probs_list[u])
            self.rewards[u].append(rewards_list[u])
        self.values.append(value)
        self.dones.append(done)

    def get_tensors(self):
        """Returns all stored data as PyTorch tensors."""
        obs_t    = [torch.FloatTensor(np.array(self.obs[u])).to(DEVICE)
                    for u in range(N_UAVS)]
        acts_t   = [torch.LongTensor(self.actions[u]).to(DEVICE)
                    for u in range(N_UAVS)]
        lps_t    = [torch.stack(self.log_probs[u]).to(DEVICE)
                    for u in range(N_UAVS)]
        rews_t   = [torch.FloatTensor(self.rewards[u]).to(DEVICE)
                    for u in range(N_UAVS)]
        vals_t   = torch.stack(self.values).squeeze(-1).to(DEVICE)
        dones_t  = torch.FloatTensor(self.dones).to(DEVICE)
        return obs_t, acts_t, lps_t, rews_t, vals_t, dones_t

## 5. GAE — Generalized Advantage Estimation

In [ ]:
def compute_gae(rewards_list, values, dones, last_value):
    """
    Computes advantages and target returns using GAE.

    GAE formula:
      δ_t   = r_t + γ × V(s_{t+1}) - V(s_t)   (TD error)
      Â_t   = Σ (γλ)^k × δ_{t+k}              (advantage)
      R_t   = Â_t + V(s_t)                     (target return)

    Args:
        rewards_list : list of reward tensors per UAV  [(T,), (T,)]
        values       : critic values tensor            (T,)
        dones        : done flags tensor               (T,)
        last_value   : V(s_T) bootstrap value         scalar tensor

    Returns:
        advantages   : (T,)  normalised advantage estimates
        returns      : (T,)  target returns for critic update
    """
    T            = len(values)
    advantages   = torch.zeros(T).to(DEVICE)
    gae          = 0.0

    # Average rewards across UAVs for shared advantage
    mean_rewards = torch.stack(rewards_list).mean(dim=0)  # (T,)

    # Append bootstrap value
    values_ext   = torch.cat([values, last_value.unsqueeze(0)])

    for t in reversed(range(T)):
        mask        = 1.0 - dones[t]
        delta       = mean_rewards[t] + GAMMA * values_ext[t+1] * mask - values_ext[t]
        gae         = delta + GAMMA * GAE_LAMBDA * mask * gae
        advantages[t] = gae

    returns = advantages + values

    # Normalise advantages
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    return advantages.detach(), returns.detach()

## 6. PPO Update

In [ ]:
def ppo_update(buffer):
    """
    Performs K_EPOCHS of PPO-Clip updates on both actors and the critic.

    Actor loss (PPO-Clip, Eq. 17 from paper):
      ratio   = exp(log_prob_new - log_prob_old)
      L_CLIP  = min(ratio × Â, clip(ratio, 1-ε, 1+ε) × Â)
      loss    = -L_CLIP - entropy_coeff × entropy

    Critic loss:
      L_VF = mean( (V(s_t) - R_t)^2 )
    """
    obs_t, acts_t, old_lps_t, rews_t, vals_t, dones_t = buffer.get_tensors()

    # Bootstrap value for GAE
    with torch.no_grad():
        last_joint = torch.cat([
            torch.FloatTensor(env._get_obs(0)).to(DEVICE),
            torch.FloatTensor(env._get_obs(1)).to(DEVICE)
        ]).unsqueeze(0)
        last_value = critic(last_joint).squeeze()

    advantages, returns = compute_gae(rews_t, vals_t, dones_t, last_value)

    actor_losses  = []
    critic_losses = []

    for _ in range(K_EPOCHS):

        # ── Actor update (per UAV) ──────────────────────────────────────────
        for u in range(N_UAVS):
            new_lps, entropy = actors[u].get_log_prob_entropy(
                obs_t[u], acts_t[u]
            )
            ratio       = torch.exp(new_lps - old_lps_t[u].detach())
            surr1       = ratio * advantages
            surr2       = torch.clamp(ratio, 1 - EPS_CLIP,
                                      1 + EPS_CLIP) * advantages
            actor_loss  = -torch.min(surr1, surr2).mean() \
                          - ENTROPY_COEFF * entropy

            actor_optimizers[u].zero_grad()
            actor_loss.backward()
            nn.utils.clip_grad_norm_(actors[u].parameters(), 0.5)
            actor_optimizers[u].step()
            actor_losses.append(actor_loss.item())

        # ── Critic update ───────────────────────────────────────────────────
        joint_obs    = torch.cat([obs_t[0], obs_t[1]], dim=1)  # (T, 34)
        values_pred  = critic(joint_obs).squeeze()             # (T,)
        critic_loss  = nn.MSELoss()(values_pred, returns)

        critic_optimizer.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(critic.parameters(), 0.5)
        critic_optimizer.step()
        critic_losses.append(critic_loss.item())

    return np.mean(actor_losses), np.mean(critic_losses)

## 7. Training Loop

In [ ]:
buffer              = RolloutBuffer()
episode_rewards     = []
episode_discovered  = []   # infected sectors found per episode
actor_loss_log      = []
critic_loss_log     = []
recent_rewards      = deque(maxlen=20)

for episode in range(1, N_EPISODES + 1):

    obs          = env.reset()
    buffer.clear()
    ep_reward    = 0.0

    for t in range(env.T):

        # ── Get critic value for joint state ─────────────────────────────
        joint_obs = torch.cat([
            torch.FloatTensor(obs[0]).to(DEVICE),
            torch.FloatTensor(obs[1]).to(DEVICE)
        ]).unsqueeze(0)

        with torch.no_grad():
            value = critic(joint_obs)

        # ── Sample actions from actors ────────────────────────────────────
        actions   = []
        log_probs = []
        for u in range(N_UAVS):
            obs_t = torch.FloatTensor(obs[u]).to(DEVICE)
            with torch.no_grad():
                action, lp = actors[u].get_action(obs_t)
            actions.append(action)
            log_probs.append(lp)

        # ── Step environment ──────────────────────────────────────────────
        next_obs, rewards, done, info = env.step(actions)

        # ── Store in buffer ───────────────────────────────────────────────
        buffer.store(
            obs_list      = obs,
            actions_list  = actions,
            log_probs_list= log_probs,
            rewards_list  = rewards,
            value         = value,
            done          = float(done)
        )

        ep_reward += sum(rewards)
        obs        = next_obs

        if done:
            break

    # ── PPO update after each episode ─────────────────────────────────────
    a_loss, c_loss = ppo_update(buffer)

    # ── Logging ───────────────────────────────────────────────────────────
    n_discovered = int((env.uav_status == 1).sum())  # known infected sectors
    episode_rewards.append(ep_reward)
    episode_discovered.append(n_discovered)
    actor_loss_log.append(a_loss)
    critic_loss_log.append(c_loss)
    recent_rewards.append(ep_reward)

    if episode % LOG_INTERVAL == 0:
        avg = np.mean(recent_rewards)
        print(f'Ep {episode:>4}/{N_EPISODES}  '
              f'Reward: {ep_reward:>8.2f}  '
              f'Avg(20): {avg:>8.2f}  '
              f'Infected found: {n_discovered:>2}  '
              f'A_loss: {a_loss:.4f}  '
              f'C_loss: {c_loss:.4f}')

    # ── Save checkpoint ───────────────────────────────────────────────────
    if episode % SAVE_INTERVAL == 0:
        for u in range(N_UAVS):
            torch.save(actors[u].state_dict(), f'actor{u}_ep{episode}.pth')
        torch.save(critic.state_dict(), f'critic_ep{episode}.pth')
        print(f'  → Checkpoint saved at episode {episode}')

print('\nTraining complete.')

## 8. Training Curves

In [ ]:
def moving_avg(data, window=20):
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('MAPPO Training — UAV Crop Disease Monitoring', fontsize=14)

# Reward
axes[0,0].plot(episode_rewards, alpha=0.4, color='steelblue', label='Raw')
axes[0,0].plot(moving_avg(episode_rewards), color='steelblue', label='Avg(20)')
axes[0,0].set_title('Total Reward per Episode')
axes[0,0].set_xlabel('Episode')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Infected sectors discovered
axes[0,1].plot(episode_discovered, alpha=0.4, color='tomato', label='Raw')
axes[0,1].plot(moving_avg(episode_discovered), color='tomato', label='Avg(20)')
axes[0,1].set_title('Infected Sectors Discovered per Episode')
axes[0,1].set_xlabel('Episode')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Actor loss
axes[1,0].plot(moving_avg(actor_loss_log), color='darkorange')
axes[1,0].set_title('Actor Loss (moving avg)')
axes[1,0].set_xlabel('Episode')
axes[1,0].grid(True, alpha=0.3)

# Critic loss
axes[1,1].plot(moving_avg(critic_loss_log), color='mediumseagreen')
axes[1,1].set_title('Critic Loss (moving avg)')
axes[1,1].set_xlabel('Episode')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved: training_curves.png')

## 9. Save Final Weights

In [ ]:
for u in range(N_UAVS):
    torch.save(actors[u].state_dict(), f'actor{u}_final.pth')
    print(f'Saved: actor{u}_final.pth')

torch.save(critic.state_dict(), 'critic_final.pth')
print('Saved: critic_final.pth')
print('\nAll weights saved. Load in 03_run_simulation.ipynb to evaluate.')